# Use case 1: ECPDS notifications to Aviso 2.0

**Ask (Laurent, Cristina):** ECPDS, the ECMWF Production Data Store, currently publishes
`dissemination` notifications to **Aviso 1.0** (etcd keys under `/ec/diss/<destination>/...`).
It could publish the same events to **Aviso 2.0** as well, so that consumers can move to the
new client while the 1.0 feed keeps running.

This notebook works through what that means end to end: the 1.x event ECPDS produces today,
the 2.x `dissemination` stream it maps onto, a pure mapping function with tests, the publishing
paths available to a Java service, the authorisation model (who may write, who may read which
destination), and the consumer side with the Aviso 2 client. Live calls go to the shared
**playground** server only.

| Setting | Value |
| --- | --- |
| Aviso 2 server | `https://aviso-playground.ecmwf.int` (override with `AVISO_BASE_URL`) |
| Streams used | `dissemination` (target of the migration), `mars` (readable MARS-like stream), `de-scm-data` (writable stand-in for the live round trip) |
| Auth | ECMWF token from `AVISO_TOKEN` or `~/.config/aviso/config.yaml` (`auth.bearer_token`) |
| Client | `pip install pyaviso requests pyyaml pandas` |
| 1.x reference | `../../aviso-deployment/schema/Bologna/aviso-prod/event_listener_schema.json` |

Sections:

1. Setup and authentication
2. What ECPDS publishes today (Aviso 1.x)
3. The Aviso 2 `dissemination` stream
4. Mapping 1.x → 2.x, with tests
5. Publishing paths for ECPDS (HTTP, `requests`, `pyaviso`, batches, dual publish)
6. Authorisation: write roles and the ECPDS destination plugin
7. Consumer side: replay, filtered watch, live round trip on `de-scm-data`
8. CLI equivalents
9. Summary and open points

## 1. Setup and authentication

Both reading and writing on the playground need an ECMWF token (from
<https://api.ecmwf.int/v1/key/>). `AVISO_TOKEN` wins; otherwise the CLI config file is read.
The token is never printed.

In [1]:
# Uncomment to install into the active kernel:
# %pip install -q pyaviso requests pyyaml pandas

import json
import os
import pathlib
import queue
import threading
import time
from datetime import datetime, timezone

import pandas as pd
import pyaviso
import requests
from IPython.display import display

BASE_URL = os.environ.get("AVISO_BASE_URL", "https://aviso-playground.ecmwf.int").rstrip("/")
CLI_CONFIG = pathlib.Path.home() / ".config" / "aviso" / "config.yaml"
REPO = pathlib.Path("../..").resolve()
LEGACY_SCHEMA = REPO / "aviso-deployment/schema/Bologna/aviso-prod/event_listener_schema.json"


def resolve_token() -> str:
    token = os.environ.get("AVISO_TOKEN")
    if token:
        return token
    if CLI_CONFIG.exists():
        import yaml
        cfg = yaml.safe_load(CLI_CONFIG.read_text(encoding="utf-8")) or {}
        token = (cfg.get("auth") or {}).get("bearer_token")
        if token:
            return token
    raise RuntimeError(f"No token: set AVISO_TOKEN or add auth.bearer_token to {CLI_CONFIG}")


def open_stream(client, event_type, *, filter, start=None, mode=None, triggers=None):
    # pyaviso 2.1.0 names the cursor `start_from=`; newer builds `from_=`.
    kwargs = {"filter": filter}
    if mode is not None:
        kwargs["mode"] = mode
    if triggers is not None:
        kwargs["triggers"] = triggers
    if start is None:
        return client.listen(event_type, **kwargs)
    try:
        return client.listen(event_type, start_from=start, **kwargs)
    except TypeError:
        return client.listen(event_type, from_=start, **kwargs)


def replay(client, event_type, *, filter, start=1):
    with open_stream(client, event_type, filter=filter, start=start, mode="replay_only") as stream:
        return list(stream)


TOKEN = resolve_token()
client = pyaviso.AvisoClient(base_url=BASE_URL, auth=pyaviso.Bearer(TOKEN))
print("pyaviso", pyaviso.__version__, "| server", client.base_url, f"| token loaded ({len(TOKEN)} chars)")
print("event types on the server:", sorted(client.schema().event_types))

pyaviso 2.1.0 | server https://aviso-playground.ecmwf.int/ | token loaded (32 chars)


event types on the server: ['cleareo_app1_data_available', 'cleareo_app1_extreme_rainfall', 'de-scm-data', 'de-scm-data-points', 'dissemination', 'extreme_event', 'mars']


## 2. What ECPDS publishes today (Aviso 1.x)

Aviso 1.x has no server-side notification API of its own: the publisher writes a key/value
pair into **etcd** and listeners poll a key prefix. The production schema below defines the
`dissemination` event: the key is built from the request fields (`base` + `stem`), and the
value is the **payload**, which for this event is the `location` of the product.

In [2]:
legacy = json.loads(LEGACY_SCHEMA.read_text(encoding="utf-8"))
diss1 = legacy["dissemination"]
etcd_endpoint = next(e for e in diss1["endpoint"] if any("etcd" in x for x in e["engine"]))
print("payload field name :", legacy["payload"])
print("etcd key template  :", etcd_endpoint["base"] + "/" + etcd_endpoint["stem"])
print("admin key template :", etcd_endpoint["admin"])
rows = [{"key": k, "handler": v[0]["type"], **{kk: vv for kk, vv in v[0].items() if kk != "type"}}
        for k, v in diss1["request"].items()]
pd.DataFrame(rows).set_index("key").fillna("")

payload field name : location
etcd key template  : /ec/diss/{destination}/date={date},target={target},class={class},expver={expver},domain={domain},time={time},stream={stream},step={step}
admin key template : /ec/admin/{date}/{destination}


,handler,canonic,default,range,values
key,,,,,
class,EnumHandler,,,,
date,DateHandler,%Y%m%d,,,
destination,StringHandler,,,,
target,StringHandler,,,,
domain,EnumHandler,,g,,
expver,IntHandler,{0:0>4},,,
step,IntHandler,,,"[0, 100000]",
stream,StringHandler,,,,
time,TimeHandler,{0:0>2},,,"[0, 6, 12, 18]"


A concrete 1.x notification, as ECPDS writes it and as the 1.x client (`pyaviso` 1.0.x) hands it
to a trigger function:

In [3]:
# What ECPDS writes (key -> value) ...
LEGACY_KEY = "/ec/diss/FOO/date=20260924,target=E1,class=od,expver=0001,domain=g,time=00,stream=oper,step=60"
LEGACY_VALUE = "/data/ecpds/FOO/E1D09240000092400001"
# ... and what a 1.x listener receives (event + request + payload field):
legacy_notification = {
    "event": "dissemination",
    "request": {"destination": "FOO", "target": "E1", "class": "od", "expver": "0001", "domain": "g",
                "date": "20260924", "time": "00", "stream": "oper", "step": "60"},
    "location": LEGACY_VALUE,
}
print(json.dumps(legacy_notification, indent=2))

{
  "event": "dissemination",
  "request": {
    "destination": "FOO",
    "target": "E1",
    "class": "od",
    "expver": "0001",
    "domain": "g",
    "date": "20260924",
    "time": "00",
    "stream": "oper",
    "step": "60"
  },
  "location": "/data/ecpds/FOO/E1D09240000092400001"
}


## 3. The Aviso 2 `dissemination` stream

Aviso 2 replaces the etcd key with an HTTP API (`POST /api/v1/notification`) and a server-owned
**schema**: the identifier fields, which are required on a watch or replay filter, and whether a
payload is mandatory. The same field names survive, so the mapping is mostly a re-shaping.

In [4]:
diss2 = client.schema_for("dissemination").schema
rows = [{"identifier": k, "type": v.get("type"), "required (watch/replay)": v.get("required", False),
         "constraint": v.get("values") or v.get("canonical_format") or v.get("range") or v.get("default") or v.get("max_length") or ""}
        for k, v in diss2["identifier"].items()]
print("payload required on notify:", diss2["payload"]["required"])
pd.DataFrame(rows).set_index("identifier")

payload required on notify: True


,type,required (watch/replay),constraint
identifier,,,
class,StringHandler,True,2
date,DateHandler,False,%Y%m%d
destination,StringHandler,True,
domain,EnumHandler,False,"[a, b, c, d, e, f, g, h, i, j, k, l, m, n, o, ..."
expver,ExpverHandler,False,0001
step,IntHandler,False,"[0, 100000]"
stream,StringHandler,False,
target,StringHandler,False,
time,TimeHandler,False,


Side by side:

| Aspect | Aviso 1.x (`dissemination`) | Aviso 2 (`dissemination` on the playground) |
| --- | --- | --- |
| Transport | etcd key/value write | `POST /api/v1/notification` (JSON, bearer token) |
| Identity of the event | key `/ec/diss/{destination}/date=…,target=…,…` | `event_type` + `identifier` object; server assigns a sequence |
| Payload | value of the key, named `location` | `payload` (any JSON; **required** on this stream) |
| Types | strings in the key (`time=00`, `expver=0001`) | typed handlers: `DateHandler` `%Y%m%d`, `TimeHandler`, `IntHandler` for `step`, `ExpverHandler` |
| Required for listeners | `destination` | `class` and `destination` (`required: true`); everything else optional |
| Delivery | client polls etcd prefix | server pushes over SSE; replay by sequence or publish time |
| Access | etcd ACL on key prefixes (`/ec/diss/` protected) | stream `auth` block: `write_roles`, `read_roles`, ECPDS destination plugin on reads |

## 4. Mapping 1.x → 2.x, with tests

A pure function ECPDS (or a bridge process reading etcd) can port: parse the 1.x key, canonicalise
the values the way the 2.x handlers expect, and emit the `notify` body. Rules applied:

* `time` `HH` → `HHMM` (`TimeHandler` accepts `HHMM`; `00` becomes `0000`).
* `expver` is kept as the 4-character string (`ExpverHandler`, default `0001`).
* `step` becomes an integer (`IntHandler`, range 0..100000).
* the 1.x `location` value goes into `payload.location`; extra context (source system,
  legacy key) rides along in the payload since the 2.x schema does not restrict payload shape.

In [5]:
import re

KEY_RE = re.compile(r"^/ec/diss/(?P<destination>[^/]+)/(?P<stem>.+)$")


def parse_legacy_key(key: str) -> dict:
    m = KEY_RE.match(key)
    if not m:
        raise ValueError(f"not a 1.x dissemination key: {key}")
    fields = dict(part.split("=", 1) for part in m.group("stem").split(","))
    fields["destination"] = m.group("destination")
    return fields


def canonical_time(value) -> str:
    v = str(value)
    if len(v) <= 2:
        return f"{int(v):02d}00"
    return f"{int(v):04d}"


def to_aviso2_notification(request: dict, location, *, event_type="dissemination", source="ecpds") -> dict:
    ident = {
        "class": str(request["class"]),
        "destination": str(request["destination"]),
        "date": str(request["date"]),
        "time": canonical_time(request["time"]),
        "expver": f"{int(request.get('expver', 1)):04d}" if str(request.get("expver", "0001")).isdigit() else str(request["expver"]),
        "domain": str(request.get("domain", "g")),
        "stream": str(request["stream"]),
        "step": int(request["step"]),
    }
    if request.get("target") is not None:
        ident["target"] = str(request["target"])
    return {"event_type": event_type, "identifier": ident,
            "payload": {"location": location, "source": source, "legacy_key": None}}


def from_legacy_key(key: str, value: str) -> dict:
    body = to_aviso2_notification(parse_legacy_key(key), value)
    body["payload"]["legacy_key"] = key
    return body


body = from_legacy_key(LEGACY_KEY, LEGACY_VALUE)
print(json.dumps(body, indent=2))

# unit-style checks
assert body["identifier"]["destination"] == "FOO" and body["identifier"]["target"] == "E1"
assert body["identifier"]["time"] == "0000" and canonical_time("12") == "1200" and canonical_time("1800") == "1800"
assert body["identifier"]["step"] == 60 and isinstance(body["identifier"]["step"], int)
assert body["identifier"]["expver"] == "0001"
assert body["payload"]["location"] == LEGACY_VALUE
assert to_aviso2_notification(legacy_notification["request"], legacy_notification["location"])["identifier"] == {
    k: v for k, v in body["identifier"].items()}
# every 2.x identifier key we emit exists in the live schema, and required ones are present
unknown = set(body["identifier"]) - set(diss2["identifier"])
assert not unknown, unknown
assert all(k in body["identifier"] for k, v in diss2["identifier"].items() if v.get("required"))
print("mapping checks passed")

{
  "event_type": "dissemination",
  "identifier": {
    "class": "od",
    "destination": "FOO",
    "date": "20260924",
    "time": "0000",
    "expver": "0001",
    "domain": "g",
    "stream": "oper",
    "step": 60,
    "target": "E1"
  },
  "payload": {
    "location": "/data/ecpds/FOO/E1D09240000092400001",
    "source": "ecpds",
    "legacy_key": "/ec/diss/FOO/date=20260924,target=E1,class=od,expver=0001,domain=g,time=00,stream=oper,step=60"
  }
}
mapping checks passed


## 5. Publishing paths for ECPDS

ECPDS is a Java service, so the primary interface is the HTTP API. Everything below is the same
single call in different clothes; the server answers `{"status":"success","request_id":...}`.

**Plain HTTP** (what a Java `HttpClient` would send):

```bash
curl -sS -X POST "$AVISO_BASE_URL/api/v1/notification" \
  -H "Authorization: Bearer $AVISO_TOKEN" -H "Content-Type: application/json" \
  -d '{"event_type":"dissemination",
       "identifier":{"class":"od","destination":"FOO","date":"20260924","time":"0000","expver":"0001",
                     "domain":"g","stream":"oper","step":60,"target":"E1"},
       "payload":{"location":"/data/ecpds/FOO/E1D09240000092400001","source":"ecpds"}}'
```

Notes for the implementation:

* **Every identifier field of the schema must be supplied on notify**, regardless of the
  `required` flag (that flag governs watch/replay filters only). Missing fields give
  `400 INVALID_NOTIFICATION_REQUEST`.
* **Idempotency.** Aviso 2 assigns a sequence per accepted notification; re-sending the same
  identifier creates a second entry unless the stream's storage policy forbids duplicates.
  ECPDS should therefore retry only on transport errors and 5xx, never on 2xx/4xx, and keep the
  `request_id` from the response for tracing.
* **Dual publish.** During the transition, publish to 1.x first (today's consumers), then to
  2.x; a 2.x failure must not block dissemination, so log it and queue for retry.

In [6]:
def notify_http(body: dict, *, base_url=BASE_URL, token=TOKEN, timeout=15) -> requests.Response:
    # The bare HTTP call, as ECPDS would issue it from Java.
    return requests.post(f"{base_url}/api/v1/notification", json=body, timeout=timeout,
                         headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"})


def notify_pyaviso(body: dict) -> pyaviso.NotifyResponse:
    return client.notify(event_type=body["event_type"], identifier=body["identifier"], payload=body["payload"])


class DualPublisher:
    # Publish to the legacy path first, then to Aviso 2; never let the 2.x path block dissemination.
    def __init__(self, legacy_write, aviso2_client, *, retries=2, backoff=0.5):
        self.legacy_write, self.client, self.retries, self.backoff = legacy_write, aviso2_client, retries, backoff
        self.failed = []          # bodies to re-send later (a real service would persist these)

    def publish(self, key: str, value: str):
        self.legacy_write(key, value)                    # 1.x: etcd put (unchanged behaviour)
        body = from_legacy_key(key, value)
        for attempt in range(self.retries + 1):
            try:
                resp = self.client.notify(event_type=body["event_type"], identifier=body["identifier"], payload=body["payload"])
                return {"aviso2": resp.status, "request_id": resp.request_id}
            except pyaviso.HttpError as exc:
                if exc.status < 500:                     # 4xx: retrying will not help
                    self.failed.append((body, f"HTTP {exc.status} {exc.body[:120]}"))
                    return {"aviso2": f"rejected HTTP {exc.status}", "request_id": exc.request_id}
            except pyaviso.TransportError as exc:
                last = str(exc)
            time.sleep(self.backoff * (2 ** attempt))
        self.failed.append((body, "transport failure"))
        return {"aviso2": "queued for retry", "request_id": None}


etcd_log = []
publisher = DualPublisher(lambda k, v: etcd_log.append((k, v)), client)
print("DualPublisher ready; legacy writes are recorded in `etcd_log` for this demo")

DualPublisher ready; legacy writes are recorded in `etcd_log` for this demo


## 6. Authorisation: write roles and the ECPDS destination plugin

Two separate gates apply on a 2.x stream (see `aviso-server/docs/src/authentication.md`):

* **Write (notify)** is governed by the stream's `auth.write_roles`. When `write_roles` is
  omitted only admins may publish. So the ECPDS service account needs a realm/role listed in
  `write_roles` of `dissemination` on the target deployment. The ECPDS plugin never runs on
  notify.
* **Read (watch/replay)** on `dissemination` is additionally gated by the **ECPDS destination
  plugin** (`aviso-server/aviso-ecpds`): the server calls
  `GET <ecpds>/ecpds/v1/destination/list?id=<username>` and only lets the user replay or watch
  destinations in their list. That is why the filter must name a `destination`, and why a
  token without any ECPDS destination gets `403 FORBIDDEN` even though it is valid. The same
  access rule 1.x enforces via etcd ACLs on `/ec/diss/<destination>` is thus preserved.

The two cells below show both gates on the live playground with this notebook's token.

In [7]:
def show_http_error(label, exc):
    try:
        body = json.loads(exc.body) if exc.body else {}
    except json.JSONDecodeError:
        body = {"message": exc.body}
    print(f"{label}: HTTP {exc.status} {body.get('code')}: {body.get('message')}  (request_id={exc.request_id})")


# Read gate: replay dissemination for one destination
try:
    got = replay(client, "dissemination", filter={"class": "od", "destination": "FOO"}, start=1)
    print(f"dissemination replay: {len(got)} notification(s) (this token has ECPDS access to destination FOO)")
except pyaviso.HttpError as exc:
    show_http_error("dissemination replay", exc)

dissemination replay: HTTP 403 FORBIDDEN: User does not have required read role for this stream  (request_id=0d2ba3e3-7a67-483b-9b3c-e254fced39d7)


In [8]:
# Write gate across all streams: an EMPTY notify body is sent to each stream. 403 = this token has no
# write role there; 400 = the write gate passed and only schema validation failed. Nothing is published.
rows = []
for ev in sorted(client.schema().event_types):
    r = notify_http({"event_type": ev, "identifier": {}, "payload": {}})
    detail = r.json()
    rows.append({"stream": ev, "http": r.status_code, "verdict": "write allowed" if r.status_code == 400 else "no write role" if r.status_code == 403 else "?",
                 "server message": detail.get("details") or detail.get("message")})
pd.DataFrame(rows).set_index("stream")

,http,verdict,server message
stream,,,
cleareo_app1_data_available,403,no write role,User does not have required write role for thi...
cleareo_app1_extreme_rainfall,403,no write role,User does not have required write role for thi...
de-scm-data,400,write allowed,Required field 'experiment' missing for notify...
de-scm-data-points,400,write allowed,Required field 'point_cloud' missing for notif...
dissemination,403,no write role,User does not have required write role for thi...
extreme_event,400,write allowed,Required field 'region' missing for notify ope...
mars,403,no write role,User does not have required write role for thi...


In [9]:
# Write gate: try to publish the mapped ECPDS event on `dissemination`. Recorded, not required to succeed.
diss_outcome = {}
try:
    resp = notify_pyaviso(body)
    diss_outcome = {"status": resp.status, "request_id": resp.request_id}
    print("dissemination notify accepted:", diss_outcome)
except pyaviso.HttpError as exc:
    diss_outcome = {"http": exc.status, "request_id": exc.request_id}
    show_http_error("dissemination notify", exc)

dissemination notify: HTTP 403 FORBIDDEN: User does not have required write role for this stream  (request_id=c31a7b6b-8c37-430b-bbc9-b4cb09fb4e46)


## 7. Consumer side: replay, filtered watch, live round trip on `de-scm-data`

A consumer of the migrated feed does exactly what a 1.x listener did, with a filter instead of
a key prefix:

```python
with client.listen("dissemination", filter={"class": "od", "destination": "FOO", "step": 60}) as stream:
    for n in stream:
        fetch(n.payload["location"])
```

This token can read `mars` but may write neither `mars` nor `dissemination` on the playground
(section 6), so the live demonstration uses **`de-scm-data`**, the DestinE data-availability
stream this token may publish to. Its shape is the same idea as an ECPDS event: identifiers
(`experiment`, `date`, `time`) plus a required payload with the location (`output_path`). The
test notifications use experiment `ecpdsuc1` and a payload note so they cannot be mistaken for
real data. `PUBLISH_TEST` is on because the playground is a sandbox; set it to `False` to only
read.

In [10]:
PUBLISH_TEST = True
DEMO_EVENT = "de-scm-data"
TEST_EXPERIMENT = "ecpdsuc1"
now = datetime.now(timezone.utc)
today = now.strftime("%Y%m%d")


def ecpds_to_demo_stream(request: dict, location: str) -> dict:
    # Same mapper idea as section 4, shaped for the writable demo stream: the MARS keys that the
    # demo schema does not have travel inside the payload so nothing is lost.
    body = to_aviso2_notification(request, location)
    return {"event_type": DEMO_EVENT,
            "identifier": {"experiment": TEST_EXPERIMENT, "date": body["identifier"]["date"], "time": body["identifier"]["time"]},
            "payload": {"output_path": location, "note": "ecpds-uc1-notebook-test", "source": "ecpds-dual-publish-demo",
                        "mars": body["identifier"]}}


demo_body = ecpds_to_demo_stream(legacy_notification["request"], LEGACY_VALUE)
demo_body["identifier"]["time"] = now.strftime("%H%M")          # unique per run
print(json.dumps(demo_body, indent=2))

if PUBLISH_TEST:
    # 1) bare HTTP, as ECPDS would do it from Java
    r = notify_http(demo_body)
    print("HTTP notify ->", r.status_code, r.json())
    # 2) pyaviso
    resp = notify_pyaviso(demo_body)
    print("pyaviso notify ->", resp.status, resp.request_id, resp.processed_at)
    # 3) a small batch with notify_many (two more steps of the same run)
    batch = []
    for step in (63, 66):
        b = ecpds_to_demo_stream({**legacy_notification["request"], "step": step}, LEGACY_VALUE.replace("E1D", f"E{step}D"))
        b["identifier"]["time"] = demo_body["identifier"]["time"]
        batch.append(b)
    for res in client.notify_many(batch):
        print(f"notify_many[{res.index}] ->", "ok " + res.response.request_id if res.ok else f"failed: {res.error}")
else:
    print("PUBLISH_TEST is False: nothing published")

{
  "event_type": "de-scm-data",
  "identifier": {
    "experiment": "ecpdsuc1",
    "date": "20260924",
    "time": "1538"
  },
  "payload": {
    "output_path": "/data/ecpds/FOO/E1D09240000092400001",
    "note": "ecpds-uc1-notebook-test",
    "source": "ecpds-dual-publish-demo",
    "mars": {
      "class": "od",
      "destination": "FOO",
      "date": "20260924",
      "time": "0000",
      "expver": "0001",
      "domain": "g",
      "stream": "oper",
      "step": 60,
      "target": "E1"
    }
  }
}
HTTP notify -> 200 {'status': 'success', 'request_id': '907a603d-1d54-4223-ade1-9480f5e45ad6', 'processed_at': '2026-09-24T15:38:02Z'}
pyaviso notify -> success b55e1c00-1c60-49db-8332-e2af7bf643ed 2026-09-24T15:38:02Z
notify_many[0] -> ok 5aa6916d-edec-4a66-a6fb-cec3cba4963a
notify_many[1] -> ok 0b985d5e-2026-43de-9d2d-8d3bf1b6b327


In [11]:
# Replay what this notebook (and earlier runs) published: filter on the test experiment, cursor = start of today (publish time).
since = now.strftime("%Y-%m-%dT00:00:00Z")
history = replay(client, DEMO_EVENT, filter={"experiment": TEST_EXPERIMENT}, start=since)
print(f"{len(history)} test notification(s) on {DEMO_EVENT} for experiment {TEST_EXPERIMENT} published since {since}")
this_run = [n for n in history if n.identifier.get("time") == demo_body["identifier"]["time"]]
print(f"published {4 if PUBLISH_TEST else 0} notification(s) with the identifier {demo_body['identifier']} in this run, "
      f"{len(this_run)} retained -> the stream keeps ONE entry per identical identifier (latest wins).")
rows = [{"sequence": n.sequence, **n.identifier, "output_path": (n.payload or {}).get("output_path"),
         "step (from payload)": ((n.payload or {}).get("mars") or {}).get("step"), "published": (n.cloudevent or {}).get("time")}
        for n in history]
pd.DataFrame(rows).tail(6)

2 test notification(s) on de-scm-data for experiment ecpdsuc1 published since 2026-09-24T00:00:00Z
published 4 notification(s) with the identifier {'experiment': 'ecpdsuc1', 'date': '20260924', 'time': '1538'} in this run, 1 retained -> the stream keeps ONE entry per identical identifier (latest wins).


,sequence,date,experiment,time,output_path,step (from payload),published
0,31,20260924,ecpdsuc1,1537,/data/ecpds/FOO/E1D09240000092400001,60,2026-09-24T15:37:14.794892428Z
1,39,20260924,ecpdsuc1,1538,/data/ecpds/FOO/E63D09240000092400001,63,2026-09-24T15:38:02.209185139Z


**Observation that matters for ECPDS.** Four notifications with the *same identifier* were
published above (bare HTTP, `pyaviso`, and two via `notify_many`, differing only in payload), yet
the replay returns a single entry: on this stream identical identifiers collapse to one stored
notification (the stream's storage policy has no `allow_duplicates`; compare `extreme_event`, which
keeps duplicates). Consequence: everything that distinguishes two ECPDS products (`step`,
`target`, `destination`, ...) **must be in the identifier**, as it is in the 2.x `dissemination`
schema, not only in the payload. It also means an ECPDS retry of the same event is harmless.

In [12]:
# The CloudEvent envelope a consumer receives (id = <event_type>@<sequence>, dataschema, publish time)
if history:
    print(json.dumps(history[-1].cloudevent, indent=2))

{
  "data": {
    "identifier": {
      "date": "20260924",
      "experiment": "ecpdsuc1",
      "time": "1538"
    },
    "payload": {
      "mars": {
        "class": "od",
        "date": "20260924",
        "destination": "FOO",
        "domain": "g",
        "expver": "0001",
        "step": 63,
        "stream": "oper",
        "target": "E1",
        "time": "0000"
      },
      "note": "ecpds-uc1-notebook-test",
      "output_path": "/data/ecpds/FOO/E63D09240000092400001",
      "source": "ecpds-dual-publish-demo"
    }
  },
  "datacontenttype": "application/json",
  "dataschema": "https://aviso-playground.ecmwf.int/schema/de-scm-data",
  "id": "de-scm-data@39",
  "source": "https://aviso-playground.ecmwf.int",
  "specversion": "1.0",
  "time": "2026-09-24T15:38:02.209185139Z",
  "type": "int.ecmwf.aviso.de-scm-data"
}


### Live round trip: watch, then publish while the watch is open

The worker thread blocks on the SSE stream and queues notifications; this thread publishes
one event through the `DualPublisher` two seconds after the watch is up, handles what arrives,
and closes the stream at the deadline. The filter names the experiment and today's date.

In [13]:
def watch_for(client, event_type, *, filter, seconds=15, max_events=1, on_open=None, start=None):
    stream = open_stream(client, event_type, filter=filter, start=start)
    q = queue.Queue()

    def pump():
        try:
            for n in stream:
                q.put(n)
        except Exception as exc:
            q.put(exc)
        finally:
            q.put(None)

    threading.Thread(target=pump, daemon=True).start()
    if on_open:
        threading.Timer(2.0, on_open).start()
    got, deadline, t0 = [], time.monotonic() + seconds, time.monotonic()
    while time.monotonic() < deadline:
        try:
            item = q.get(timeout=min(1.0, max(deadline - time.monotonic(), 0.01)))
        except queue.Empty:
            continue
        if item is None:
            break
        if isinstance(item, Exception):
            print("stream error:", type(item).__name__, item)
            break
        print(f"[{time.monotonic() - t0:5.1f}s] seq={item.sequence} {item.identifier} payload={item.payload}")
        got.append(item)
        if max_events and len(got) >= max_events:
            break
    stream.close()
    print(f"watch closed after {time.monotonic() - t0:.1f}s with {len(got)} notification(s)")
    return got


class DemoDualPublisher(DualPublisher):
    # Same retry/queue behaviour, 2.x side shaped for the writable demo stream.
    def publish(self, key: str, value: str):
        self.legacy_write(key, value)
        body2 = ecpds_to_demo_stream(parse_legacy_key(key), value)
        body2["identifier"]["time"] = datetime.now(timezone.utc).strftime("%H%M")
        body2["payload"]["legacy_key"] = key
        try:
            resp = self.client.notify(**body2)
            return {"aviso2": resp.status, "request_id": resp.request_id}
        except pyaviso.HttpError as exc:
            self.failed.append((body2, f"HTTP {exc.status}"))
            return {"aviso2": f"rejected HTTP {exc.status}", "request_id": exc.request_id}


demo_publisher = DemoDualPublisher(lambda k, v: etcd_log.append((k, v)), client)


def publish_via_dual_publisher():
    if PUBLISH_TEST:
        key = f"/ec/diss/FOO/date={today},target=E1,class=od,expver=0001,domain=g,time=00,stream=oper,step=60"
        print("  dual publisher ->", demo_publisher.publish(key, LEGACY_VALUE))


live = watch_for(client, DEMO_EVENT, filter={"experiment": TEST_EXPERIMENT, "date": today}, seconds=15,
                 on_open=publish_via_dual_publisher)
print("legacy writes recorded:", len(etcd_log), "| 2.x failures queued:", len(demo_publisher.failed))

  dual publisher -> {'aviso2': 'success', 'request_id': 'be6ae8fd-6ead-4110-b549-148de72ef943'}
[  2.0s] seq=40 {'date': '20260924', 'experiment': 'ecpdsuc1', 'time': '1538'} payload={'legacy_key': '/ec/diss/FOO/date=20260924,target=E1,class=od,expver=0001,domain=g,time=00,stream=oper,step=60', 'mars': {'class': 'od', 'date': '20260924', 'destination': 'FOO', 'domain': 'g', 'expver': '0001', 'step': 60, 'stream': 'oper', 'target': 'E1', 'time': '0000'}, 'note': 'ecpds-uc1-notebook-test', 'output_path': '/data/ecpds/FOO/E1D09240000092400001', 'source': 'ecpds-dual-publish-demo'}
watch closed after 2.1s with 1 notification(s)
legacy writes recorded: 1 | 2.x failures queued: 0


## 8. CLI equivalents

```bash
export AVISO_BASE_URL=https://aviso-playground.ecmwf.int AVISO_TOKEN=...
aviso schema get dissemination
aviso notify 'event=de-scm-data,experiment=ecpdsuc1,date=20260924,time=0000,data={"output_path":"/data/ecpds/FOO/E1D09240000092400001","note":"ecpds-uc1-notebook-test"}'
aviso replay --event de-scm-data --identifiers '{"experiment":"ecpdsuc1"}' --from 2026-09-24 --json
aviso replay --event mars --identifiers '{"class":"od","step":60}' --from 1 --json      # read-only stream for this token
aviso listen --event dissemination --identifiers '{"class":"od","destination":"FOO","step":60}' --json
```

In [14]:
!aviso --version
!aviso schema get dissemination --base-url {BASE_URL} | head -40

aviso 2.1.0


{
  "event_type": "dissemination",
  "schema": {
    "identifier": {
      "class": {
        "max_length": 2,
        "required": true,
        "type": "StringHandler"
      },
      "date": {
        "canonical_format": "%Y%m%d",
        "required": false,
        "type": "DateHandler"
      },
      "destination": {
        "max_length": null,
        "required": true,
        "type": "StringHandler"
      },
      "domain": {
        "required": false,
        "type": "EnumHandler",
        "values": [
          "a",
          "b",
          "c",
          "d",
          "e",
          "f",
          "g",
          "h",
          "i",
          "j",
          "k",
          "l",
          "m",
          "n",
          "o",
          "p",
          "q",


## 9. Summary and open points

What this notebook established on the playground:

* The 2.x `dissemination` schema carries every field of the 1.x key, so a **stateless
  mapper** (section 4) is enough; `location` moves into the payload.
* Publishing is one HTTP `POST`; `pyaviso` and `notify_many` wrap the same call. The
  dual-publish wrapper isolates 1.x dissemination from any 2.x failure.
* Access is per stream and per direction. The ECMWF token used here **reads `mars`**, may
  **write `de-scm-data`, `de-scm-data-points` and `extreme_event`**, but gets **403 on
  `dissemination` and `mars` writes** (no write role) and **403 on `dissemination` reads**
  (ECPDS destination plugin: no destination for this user). Section 6 has the table.

To take to Laurent and Cristina:

1. Which realm/role the ECPDS service account will carry, and add it to `write_roles` of
   `dissemination` on the target deployment (Bologna prod values in `aviso-deployment`).
2. Whether ECPDS should publish natively (Java `HttpClient`, section 5) or whether a small
   **bridge** should tail the 1.x etcd prefix `/ec/diss/` and forward (zero change in ECPDS,
   one extra hop).
3. Duplicate policy for `dissemination` in 2.x: on the playground streams without
   `allow_duplicates` keep one entry per identifier (section 7), so ECPDS retries are safe as
   long as every distinguishing key stays in the identifier.
4. Canonical `time` format to standardise on (`0000` here) so 1.x and 2.x consumers agree.

## Related Polytope examples

The DestinE Polytope examples repository (`~/Git/polytope/polytope-examples`, GitHub
`destination-earth-digital-twins/polytope-examples`) contains no Aviso code, but it is the
reference for the **data side** of every Aviso workflow: once a notification names a field, these
notebooks show how to fetch exactly that field (or a feature cut from it) with earthkit-data.

| Need | Example |
| --- | --- |
| Full Extremes-DT field, regional `area` and `grid` options | `extremes-dt/full-field/extremes-dt-earthkit-example-domain.ipynb`, `extremes-dt/full-field-post-processing/extremes-dt-earthkit-serverside-interpolation.ipynb` |
| Feature extraction (timeseries, bounding box, polygon/country, vertical profile, trajectory, wave) | `extremes-dt/feature-extraction/*.ipynb` |
| On-Demand Extremes-DT (DEODE) output: `dataset: on-demand-extremes-dt`, `expver: 0099`, per-run `georef` | `on-demand-extremes-dt/full-field/` and `on-demand-extremes-dt/feature-extraction/` |
| DESP token (`~/.polytopeapirc`) | `desp-authentication.py` |

Operational limits from that README: 50 requests per second and at most 5 concurrent downloads per
user; data are served from LUMI (`polytope.lumi.apps.dte.destination-earth.eu`) and MN5
(`polytope.mn5.apps.dte.destination-earth.eu`). Relative dates such as `"date": "-14"` are accepted.
